### Instruction Fine Tuning with PEFT - QLoRA

In this type we provide Instruction along with the input (QnA) to the model.

##### 1. Understanding Model for fine-tuning

In [1]:
# run on Google Colab
%%capture
! pip install --upgrade pyarrow datasets trl torchao bitsandbytes

In [2]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "false"
import torch
from transformers import pipeline

In [3]:
# using very small model
# model_id = "arnir0/Tiny-LLM"
model_id="HuggingFaceTB/SmolLM2-135M"

In [4]:

llm_model = pipeline(
    "text-generation",
    model=model_id,
    dtype=torch.bfloat16 )

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

In [5]:
# interrogatory query
query="When will the Cooling Tower at the Stars Hollow plant be finished?"
# answer: "The estimated completion or milestone date is 8th Dec 2026."

In [6]:
# let's check what model gives now
llm_model(query)[0]["generated_text"]

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'When will the Cooling Tower at the Stars Hollow plant be finished?\n\nThe cooling tower is scheduled to be finished by the end of 2015. The cooling tower is a device that is used to cool the water in the boiler. The water is boiled in the boiler and then the steam is passed through a turbine. The turbine then turns a generator and produces electricity. The cooling tower also helps to reduce the amount of nitrogen oxide that is produced during the process of steam generation.\n\n\nWhat are the advantages of the cooling tower?\n\nThe cooling tower has many advantages. The main advantage is that it helps to reduce the amount of nitrogen oxide that is produced during the steam generation process. This helps to reduce the air pollution that is emitted into the atmosphere. The cooling tower also helps to cool the water in the boiler. This helps to reduce the amount of water that is necessary for the steam to be produced. The cooling tower can be made from several materials, including concre

weird, but expected, let's try fine tuning using LoRA

##### 2. Loading Model for fine-tuning

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, PeftModel, PeftConfig
from trl import SFTConfig, SFTTrainer


In [8]:
# loading tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# setting padding
tokenizer.pad_token = tokenizer.eos_token

# setting chat template as our dataset is chat like dataset
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '### ' + message['role'].upper() + ':\n' + message['content'] + '\n\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '### ASSISTANT:\n' }}"
    "{% endif %}"
)

In [9]:
# need to specifiy bits and byte configs to load the model with quantization
# Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Enables the compression of the base model's weights into 4-bit precision, reducing VRAM usage by approximately 4x compared to standard 16-bit.
    bnb_4bit_quant_type="nf4", # Defines the mathematical data type used for storage; "nf4" (Normal Float 4) is specialized for normally distributed neural network weights and offers better accuracy than standard 4-bit integers.
    bnb_4bit_use_double_quant=True, # Performs a second round of quantization on the "quantization constants" themselves, saving an extra 0.4 bits per parameter on average without losing accuracy.
    bnb_4bit_compute_dtype=torch.float16 # Specifies the data type used for the actual matrix multiplications during training; using torch.bfloat16 ensures the math stays precise and fast on modern GPUs (like A100s or T4s).
)

In [10]:
# loading model

model = AutoModelForCausalLM.from_pretrained(model_id,
                                             quantization_config=bnb_config,
                                             trust_remote_code=True, # Tiny models often use custom scripts
                                             device_map="auto"
                                             )

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [11]:
# safety and compatibility check

model = prepare_model_for_kbit_training(model)

##### 3. LoRA Configurations

In [12]:
##### 3. LoRA Config

peft_lora_config = LoraConfig(
    task_type="CAUSAL_LM", # Informs the PEFT library about the model's objective; "CAUSAL_LM" is used for standard text generation models
    r=16, # (Rank): The dimension of the low-rank matrices; a higher value allows the model to learn more complex patterns but increases the number of trainable parameters.
    lora_alpha=32, # A scaling factor for the LoRA weights that determines how much influence the adapters have over the original model's predictions (usually set to 2*r).
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # A list specifying which internal layers of the Transformer architecture will have LoRA adapters attached to them.
    lora_dropout=0.05,  # The probability of randomly "dropping out" (setting to zero) some neurons in the LoRA layers during training to prevent overfitting.
    bias="none" # Specifies if the "bias" parameters of the model should be trained; setting to "none" ensures only the LoRA weights are updated for maximum efficiency.
)

In [13]:
# loading peft type model

peft_model = get_peft_model(model, peft_lora_config)

##### 4. Defining SFTConfig and Trainer for Training

In [14]:
# adding sft config
# modified the paratmers as per local system configuration but ideally we should set as per the availble system
sft_config = SFTConfig(
    output_dir="./tiny-llm-ift-peft-qlora-training",  # The file path where the model checkpoints, tokenizer files, and training logs will be saved.
    dataset_text_field="messages", # Tells the trainer which specific column name in your dataset contains the text to be trained on.
    packing=False, # If true, it combines multiple short examples into a single sequence of
    per_device_train_batch_size=4, # The number of training examples processed simultaneously on a single GPU. ideal is 4
    gradient_accumulation_steps=4, # The number of steps to wait (accumulating gradients) before performing a single weight update, effectively increasing the "total" batch size. ideal is 4
    learning_rate=2e-5, # The "step size" the optimizer takes to minimize the error; too high may cause instability, too low may be too slow. 2e-5 ideal
    num_train_epochs=1, # The total number of times the model will see the entire training dataset.
    save_steps=100, # How often (in training steps) the trainer saves a backup checkpoint of the model to the output directory.
    logging_steps=10, # How frequently the training progress (loss, learning rate, etc.) is printed to the console or log.
    lr_scheduler_type="cosine", # Defines how the learning rate changes over time (e.g., "cosine" smoothly reduces the rate towards the end of training).
    #optim="paged_adamw_32bit", # The specific optimization algorithm used; paged_adamw_32bit is memory-efficient and ideal for QLoRA.
    report_to="none", # Specifies external platforms (like Weights & Biases or TensorBoard) where training metrics should be sent for visualization.
    #use_cpu=True,
)

loading dataset

In [15]:
# mounting google drive such that we can use the files for ingest
from google.colab import drive
drive.mount('/content/drive')

dataset_file_path = "/content/drive/MyDrive/Colab Notebooks/instruction_fine_tuning/instruction_fine_tuning_training_data.json"

Mounted at /content/drive


In [16]:
# local file path
# dataset_file_path = os.path.join(os.getcwd(), "instruction_fine_tuning_training_data.jsonl")

In [17]:
# laoding dataset
from datasets import load_dataset
#dataset = Dataset.from_text(dataset_file_path)
dataset = load_dataset("json", data_files=dataset_file_path, split="train")


Generating train split: 0 examples [00:00, ? examples/s]

In [18]:
# function to apply chat template such that model can understand
def format_for_sft(train_dataset):
    # No json.loads() needed here! load_dataset("json") already parsed it.
    formatted_text = tokenizer.apply_chat_template(
        train_dataset["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    # Add EOS token so the model learns when to stop
    return {"text": formatted_text + tokenizer.eos_token}

In [19]:
dataset = dataset.map(format_for_sft)

Map:   0%|          | 0/10124 [00:00<?, ? examples/s]

In [20]:
print(dataset[0]["text"])

### SYSTEM:
You are a professional corporate legal assistant. Answer questions based on official company records.

### USER:
Identify every instance since 2024 where the Defendant exceeded the sulfur dioxide (SO2) emission limits specified in its Title V Operating Permit for the Coal-Fired Unit 4.

### ASSISTANT:
There was one recorded exceedance on February 12, 2025, lasting 45 minutes during a cold-start procedure. The incident was reported to the state agency within the required 24-hour window.

<|endoftext|>


In [21]:
# initilizing SFTTrainer

trainer = SFTTrainer(
    model=peft_model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer
)

Tokenizing train dataset:   0%|          | 0/10124 [00:00<?, ? examples/s]

In [22]:
# let's see how many paramter trainer will be fine tuning
trainer.get_num_trainable_parameters()

4884480

In [23]:
# let's see how many paramter trainer will be fine tuning
peft_model.print_trainable_parameters()

trainable params: 4,884,480 || all params: 139,399,488 || trainable%: 3.5039


##### 5. Let's train and Save the model

In [24]:
# training the mode
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,3.511605
20,3.371016
30,3.253422
40,3.155032
50,3.017308
60,2.902123
70,2.793136
80,2.643998
90,2.552228
100,2.418571


TrainOutput(global_step=633, training_loss=1.4577003320058186, metrics={'train_runtime': 1835.9351, 'train_samples_per_second': 5.514, 'train_steps_per_second': 0.345, 'total_flos': 497912125814784.0, 'train_loss': 1.4577003320058186})

In [25]:
# saving into drive
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/instruction_fine_tuning/ift-peft-qlora-adapter")

In [26]:
# saving the model
trainer.save_model("./ift-peft-qlora-adapter")

##### 6. Let's Load and Test the model

In [27]:
adapter_path = "./ift-peft-qlora-adapter"

In [28]:
# merging with base model
adapter = PeftConfig.from_pretrained(adapter_path)

# fethcing base model name fropm adapter config
base_model_name = adapter.base_model_name_or_path


In [29]:
# loading base model and it's tokenizer
base_model = AutoModelForCausalLM.from_pretrained(base_model_name)

# loading tokenizer
base_tokenizer = AutoTokenizer.from_pretrained(adapter_path)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [30]:
# let's load the model
ft_model = PeftModel.from_pretrained(base_model, adapter_path)

In [31]:
messages =  [{"role": "system", "content": "You are a professional corporate legal assistant. Answer questions based on official company records."},
 {"role": "user", "content": query},
 ]

# we have to apply same chat template to get the result
prompt = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
   return_tensors="pt")

input_tokens = base_tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
input_tokens


{'input_ids': tensor([[ 3757, 46273,    42,   198,  2683,   359,   253,  3544,  9749,  3425,
         11173,    30, 19842,  2029,  1552,   335,  5326,  2727,  4585,    30,
           198,   198,  3757,  2038,  1754,    42,   198,  2427,   523,   260,
         48019, 18669,   418,   260, 27069, 42884,  1783,   325,  7721,    47,
           198,   198,  3757, 40800,  9814, 17321,    42,   198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [32]:
# let's grab the output token
output_tokens = ft_model.generate(
            input_ids=input_tokens["input_ids"],
            max_new_tokens=100,
            temperature="0.5",
            repetition_penalty=1.2,
            #eos_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.encode('"}')
        )

output = base_tokenizer.batch_decode(output_tokens, skip_special_tokens=True)[0]


print(output)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:23597 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


### SYSTEM:
You are a professional corporate legal assistant. Answer questions based on official company records.

### USER:
When will the Cooling Tower at the Stars Hollow plant be finished?

### ASSISTANT:
The project is scheduled to begin in 2019 and should complete by 2035.

# 实验三：编写一个函数来计算一组整型的测试数据。
## 概述

在这个实验中，我们使用Python和PyCharm已有功能开
